<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 16</h2>
<h2>MAK (Message Authentication Code)</h2>
<h3 style="color: #8B4513;">CBC-MAC, CMAC, GMAC və UF-CMA təhlükəsizlik modeli</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: MAK nədir? (10 dəq)](#xatırlatma-mak-nədir-10-dəq)
5. [Sadə MAK nümunələri (15 dəq)](#sadə-mak-nümunələri-15-dəq)
6. [CBC-MAC (15 dəq)](#cbc-mac-15-dəq)
7. [CMAC (Cipher-based MAC) (20 dəq)](#cmac-cipher-based-mac-20-dəq)
8. [GMAC (GCM Authentication) (10 dəq)](#gmac-gcm-authentication-10-dəq)
9. [MAK və rəqəmsal imza müqayisəsi (10 dəq)](#mak-və-rəqəmsal-imza-müqayisəsi-10-dəq)
10. [UF-CMA təhlükəsizlik modeli (10 dəq)](#uf-cma-təhlükəsizlik-modeli-10-dəq)
11. [İnteqrasiya edilmiş tətbiq (15 dəq)](#inteqrasiya-edilmiş-tətbiq-15-dəq)
12. [Ev tapşırığı](#ev-tapşırığı)
13. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
14. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ MAK (Message Authentication Code) anlayışını və təhlükəsizlik xüsusiyyətlərini izah edə biləcəksiniz  
✅ CBC-MAC və onun dəyişən uzunluq problemlərini başa düşəcəksiniz  
✅ CMAC (Cipher-based MAC) alqoritmini Python-da implementasiya edə biləcəksiniz  
✅ PyCryptodome kitabxanası ilə CMAC istifadə edə biləcəksiniz  
✅ MAK və rəqəmsal imza arasındakı fərqləri izah edə biləcəksiniz  
✅ UF-CMA təhlükəsizlik modelini başa düşəcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | Quraşdırma | İstifadə sahəsi |
|-----------|------------|-----------------|
| PyCryptodome | `!pip install pycryptodome` | AES-CMAC, AES-CCM |
| cryptography | `!pip install cryptography` | CMAC, HMAC |
| hashlib | Python standart kitabxanası | HMAC-SHA256 |
| hmac | Python standart kitabxanası | HMAC hesablamaları |

> 💡 **Qeyd:** PyCryptodome kitabxanası mütləq quraşdırılmalıdır. Bu kitabxana olmadan CMAC və CBC-MAC implementasiyalarını test edə bilməyəcəksiniz.

In [ ]:
# Lazımi kitabxanaların quraşdırılması

!pip install pycryptodome cryptography --quiet

print("✅ Kitabxanalar quraşdırıldı")

## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:

import sys
import os
import hmac
import hashlib
import time
import math
import random
from pathlib import Path
from collections import Counter

# PyCryptodome kitabxanası
try:
    from Crypto.Cipher import AES
    from Crypto.Random import get_random_bytes
    from Crypto.Util.Padding import pad, unpad
    from Crypto.Hash import CMAC as PyCryptoCMAC
    print("✅ PyCryptodome yüklü - AES, CMAC, GMAC istifadə edilə bilər")
    CRYPTO_AVAILABLE = True
except ImportError:
    print("❌ PyCryptodome yoxdur! Əvvəlcə quraşdırın: pip install pycryptodome")
    CRYPTO_AVAILABLE = False

# cryptography kitabxanası
try:
    from cryptography.hazmat.primitives import cmac as cryptography_cmac
    from cryptography.hazmat.primitives.ciphers import algorithms
    print("✅ cryptography yüklü - alternativ CMAC yoxlaması edilə bilər")
    CRYPTOGRAPHY_AVAILABLE = True
except ImportError:
    print("⚠️ cryptography yoxdur - alternativ implementasiyalar işləyəcək")
    CRYPTOGRAPHY_AVAILABLE = False

def xor_bytes(a: bytes, b: bytes) -> bytes:
    """İki eyni uzunluqlu byte sətrini XOR et."""
    if len(a) != len(b):
        raise ValueError("XOR üçün girişlərin uzunluğu eyni olmalıdır")
    return bytes(x ^ y for x, y in zip(a, b))

def iter_blocks(data: bytes, block_size: int = 16):
    """Mesajı sabit ölçülü bloklara böl."""
    if len(data) % block_size != 0:
        raise ValueError(f"Mesaj uzunluğu {block_size} baytın qatında olmalıdır")
    for i in range(0, len(data), block_size):
        yield data[i:i + block_size]

print(f"\n🐍 Python versiyası: {sys.version}")


### 3.2 İşçi qovluğun yaradılması

In [ ]:

# İşçi qovluğu yaradaq
if "NOTEBOOK_ROOT" not in globals():
    NOTEBOOK_ROOT = Path.cwd().resolve()

workspace = NOTEBOOK_ROOT / "crypto_workshop" / "lecture16"
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)

print(f"📁 İşçi qovluq: {workspace}")
print(f"📂 Qovluğun məzmunu: {os.listdir('.')}")


## 📖 Xatırlatma: MAK nədir? (10 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 MAK (Message Authentication Code)</h4>
<p>MAK — mesajın tamlığını və mənbə autentifikasiyasını təmin edən kriptoqrafik mexanizmdir.</p>

<p style="text-align: center; font-size: 1.2em;">$$ \text{MAK} = \text{MAC}_K(M) $$</p>

<p>burada:</p>
<ul>
    <li>$K$ — gizli açar</li>
    <li>$M$ — mesaj</li>
    <li>$\text{MAC}_K$ — MAK funksiyası</li>
</ul>

<p><b>Təhlükəsizlik xüsusiyyətləri:</b></p>
<ul>
    <li><b>Tamlıq:</b> Mesaj dəyişdirilərsə, MAK dəyişir</li>
    <li><b>Autentifikasiya:</b> Düzgün MAK yalnız açarı bilən tərəf yarada bilər</li>
    <li><b>UF-CMA:</b> Seçilmiş mesaj hücumlarına qarşı saxtalaşdırılmaya davamlı</li>
</ul>
</div>

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50; margin-top: 10px;">
<h4>📌 MAK istifadəsi</h4>
<p>Alis Boba mesaj göndərir:</p>
<ol>
    <li>Alis: $t = \text{MAC}_K(M)$ hesablayır</li>
    <li>Alis Boba $(M, t)$ göndərir</li>
    <li>Bob: $t' = \text{MAC}_K(M)$ hesablayır</li>
    <li>Bob $t$ və $t'$ teqlərini sabit-vaxtlı müqayisə ilə yoxlayır (Python-da, məsələn, <code>hmac.compare_digest(t, t')</code>)</li>
</ol>
<p>Əgər sabit-vaxtlı müqayisə uğurlu olarsa, mesaj dəyişdirilməyib və Alisdən gəlib.</p>
</div>

## 🛠️ Sadə MAK nümunələri (15 dəq)

In [ ]:

def hmac_sha256(key, message):
    """HMAC-SHA256 teqi qaytar."""
    return hmac.new(key, message, hashlib.sha256).digest()

def hmac_verify(key, message, mac):
    """HMAC-SHA256 teqini təhlükəsiz şəkildə yoxla."""
    expected_mac = hmac_sha256(key, message)
    return hmac.compare_digest(expected_mac, mac)

def hmac_demo():
    """
    Python hmac modulu ilə MAK nümayişi
    """
    print("=" * 70)
    print("🔏 HMAC (HASH-BASED MAC) NÜMAYİŞİ")
    print("=" * 70)

    # Gizli açar
    key = b'gizli_acar_123456'
    message = b'Bu vacib mesajdir'

    print(f"🔑 Açar: {key}")
    print(f"📨 Mesaj: {message}")

    # HMAC-SHA256 hesabla
    mac = hmac_sha256(key, message)

    print(f"\n🔏 HMAC-SHA256: {mac.hex()}")
    print(f"   Uzunluq: {len(mac)} bayt = {len(mac) * 8} bit")

    # Düzgün yoxlama
    print(f"\n✅ Orijinal mesaj yoxlaması: {hmac_verify(key, message, mac)}")

    modified_message = message + b"!"
    print(f"❌ Dəyişdirilmiş mesaj yoxlaması: {hmac_verify(key, modified_message, mac)}")

    return key, message, mac

hmac_demo()


### 5.1 Sadə XOR əsaslı MAK (təhlükəsiz deyil!)

In [ ]:
def simple_xor_mac(key, message):
    """
    Sadə XOR əsaslı MAK - TƏHLÜKƏSİZ DEYİL!
    Yalnız təhsil məqsədləri üçün
    """
    # Açarı mesaj uzunluğuna qədər təkrarla
    if len(key) < len(message):
        key = key * (len(message) // len(key) + 1)
    key = key[:len(message)]

    # XOR
    result = bytearray()
    for i, b in enumerate(message):
        result.append(b ^ key[i])

    # Nəticəni sabit uzunluğa endir (sadəcə nümunə üçün)
    # Bu, təhlükəsiz MAK üsulu DEYIL!
    mac = 0
    for b in result:
        mac ^= b

    return mac.to_bytes(1, 'big')

print("\n" + "-" * 70)
print("⚠️ SADƏ XOR MAC (TƏHLÜKƏSİZ DEYİL!)")
print("-" * 70)

key = b'key'
message = b'Message'
mac = simple_xor_mac(key, message)
print(f"Key: {key}, Message: {message}, MAC: {mac.hex()}")
print("🚨 Bu MAK təhlükəsiz deyil! Yalnız təhsil üçün.")

### ✍️ Çalışma 1: Sadə MAK (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Şəxsi MAK:** HMAC-SHA256 istifadə edərək öz adınız üçün MAK yaradın.

2. **Fərqli mesaj testi:** Eyni açar, fərqli mesaj üçün MAK hesablayın. Nəticələr fərqlidirmi?

3. **Fərqli açar testi:** Fərqli açar, eyni mesaj üçün MAK hesablayın. Nəticələr fərqlidirmi?

4. **Nəzəri sual:** Sadə XOR MAC-in niyə təhlükəsiz olmadığını izah edin.

In [ ]:
# Çalışma 1 - Cavablar

print("📝 ÇALIŞMA 1 CAVABLARI")
print("=" * 80)

# 1. Şəxsi MAK
print("\n1. ŞƏXSİ MAK:")
my_name = "Səməd Vəkilov"  # Öz adınızla əvəz edin
key = b'gizli_ortaq_acar'
h = hmac.new(key, my_name.encode(), hashlib.sha256)
print(f"   Ad: {my_name}")
print(f"   HMAC-SHA256: {h.hexdigest()}")

# 2. Fərqli mesaj testi
print("\n2. FƏRQLİ MESAJ TESTİ:")
msg1 = b"Message A"
msg2 = b"Message B"
h1 = hmac.new(key, msg1, hashlib.sha256).hexdigest()
h2 = hmac.new(key, msg2, hashlib.sha256).hexdigest()
print(f"   MAK(msg1): {h1}")
print(f"   MAK(msg2): {h2}")
print(f"   MAK-lar eynidir? {h1 == h2}")

# 3. Fərqli açar testi
print("\n3. FƏRQLİ AÇAR TESTİ:")
key1 = b'key1'
key2 = b'key2'
message = b"Test message"
hmac1 = hmac.new(key1, message, hashlib.sha256).hexdigest()
hmac2 = hmac.new(key2, message, hashlib.sha256).hexdigest()
print(f"   HMAC(key1): {hmac1}")
print(f"   HMAC(key2): {hmac2}")
print(f"   HMAC-lar eynidir? {hmac1 == hmac2}")

# 4. XOR MAC təhlükəsizliyi
print("\n4. XOR MAC TƏHLÜKƏSİZLİYİ:")
print("""
   XOR MAC təhlükəsiz deyil, çünki:
   • Xətti olduğu üçün asanlıqla bərpa edilə bilər
   • MAC(x) ⊕ MAC(y) = MAC(x ⊕ y) xüsusiyyəti var
   • Hücumçu mesajları dəyişdirərək saxta MAC yarada bilər
   • Təhlükəsiz MAK üçün qeyri-xətti funksiya tələb olunur (HMAC, CMAC)
""")

## 🔗 CBC-MAC (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔗 CBC-MAC</h4>
<p>CBC-MAC, CBC rejimində blok şifrə istifadə edərək MAK yaradır. Son şifrəli mətn bloku MAK kimi istifadə olunur.</p>
</div>

In [ ]:

def cbc_mac(message, key):
    """
    Təhsil məqsədli standart CBC-MAC.

    Qeyd:
    - IV həmişə 0^128 götürülür.
    - Mesaj uzunluğu blok ölçüsünün (16 bayt) qatında olmalıdır.
    - CBC-MAC eyni açar altında yalnız sabit uzunluqlu mesajlar üçün təhlükəsizdir.
    """
    if not CRYPTO_AVAILABLE:
        print("PyCryptodome yüklü deyil")
        return None

    block_size = 16  # AES blok ölçüsü

    if len(key) not in (16, 24, 32):
        raise ValueError("AES açarı 16, 24 və ya 32 bayt olmalıdır")

    if len(message) == 0 or len(message) % block_size != 0:
        raise ValueError(
            "CBC-MAC üçün mesaj boş olmamalı və 16 baytın qatında olmalıdır"
        )

    cipher = AES.new(key, AES.MODE_CBC, iv=b'\x00' * block_size)
    ciphertext = cipher.encrypt(message)

    # Son blok MAC-dır
    return ciphertext[-block_size:]

def cbc_mac_verify(message, key, mac):
    """CBC-MAC yoxlaması"""
    try:
        expected_mac = cbc_mac(message, key)
    except ValueError:
        return False
    if expected_mac is None:
        return False
    return hmac.compare_digest(mac, expected_mac)

# Test
if CRYPTO_AVAILABLE:
    print("\n" + "=" * 70)
    print("🔗 CBC-MAC TESTİ")
    print("=" * 70)

    key = get_random_bytes(16)
    message = b"0123456789ABCDEFfedcba9876543210"  # 32 bayt = 2 blok

    mac = cbc_mac(message, key)
    print(f"📨 Mesaj: {message}")
    print(f"   Uzunluq: {len(message)} bayt")
    print(f"🔏 MAC: {mac.hex()}")

    # Yoxlama
    valid = cbc_mac_verify(message, key, mac)
    print(f"✅ Yoxlama: {valid}")

    # Dəyişdirilmiş mesaj
    modified = b"1123456789ABCDEFfedcba9876543210"
    valid = cbc_mac_verify(modified, key, mac)
    print(f"❌ Dəyişdirilmiş mesaj yoxlaması: {valid}")


### 6.1 CBC-MAC-ın dəyişən uzunluq problemi

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ CBC-MAC dəyişən uzunluq problemi</h4>
<p>Sabit uzunluqlu mesajlar üçün CBC-MAC təhlükəsizdir. Lakin dəyişən uzunluqlu mesajlar üçün problem var: hücumçu iki MAK-ı birləşdirərək yeni etibarlı MAK yarada bilər.</p>
</div>

In [ ]:

def cbc_mac_length_extension_attack():
    """
    CBC-MAC-in dəyişən uzunluqlu mesajlar üçün niyə təhlükəli olduğunu göstər.

    Hücum ideyası:
    Əgər hücumçu MAC(M1) və MAC(M2) teqlərini əldə edibsə, o zaman
    F = M1 || (M2_1 ⊕ MAC(M1)) || M2_2 || ... || M2_n
    mesajı üçün etibarlı teq kimi MAC(M2)-ni təqdim edə bilər.
    """
    if not CRYPTO_AVAILABLE:
        return

    print("\n" + "-" * 70)
    print("⚠️ CBC-MAC DƏYİŞƏN UZUNLUQ HÜCUMU")
    print("-" * 70)

    key = get_random_bytes(16)
    block_size = 16

    # M1 - 1 blok, M2 - 2 blok
    m1 = b"A" * block_size
    m2 = b"B" * block_size + b"C" * block_size

    mac1 = cbc_mac(m1, key)
    mac2 = cbc_mac(m2, key)

    # Saxtalaşdırılmış yeni mesaj:
    # F = M1 || (M2_1 xor MAC(M1)) || M2_2
    m2_first = m2[:block_size]
    m2_rest = m2[block_size:]
    forged_message = m1 + xor_bytes(m2_first, mac1) + m2_rest

    # Hücumçu real sistemdə bunu HESABLAYA bilməz, sadəcə mac2-ni istifadə edər.
    # Biz burada yoxlama üçün həqiqi teqi də hesablayırıq.
    real_forged_mac = cbc_mac(forged_message, key)
    forged_tag = mac2

    print(f"📨 M1: {m1}")
    print(f"🔏 MAC(M1): {mac1.hex()}")
    print(f"📨 M2: {m2}")
    print(f"🔏 MAC(M2): {mac2.hex()}")

    print(f"\n🧪 Saxta mesajın uzunluğu: {len(forged_message)} bayt")
    print(f"🕵️ Hücumçunun təqdim etdiyi teq: {forged_tag.hex()}")
    print(f"✅ Həqiqi MAC(saxta_mesaj): {real_forged_mac.hex()}")
    print(f"\n🔍 Saxtalaşdırma uğurludur? {forged_tag == real_forged_mac}")

if CRYPTO_AVAILABLE:
    cbc_mac_length_extension_attack()


### ✍️ Çalışma 2: CBC-MAC (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **CBC-MAC testi:** CBC-MAC ilə bir mesaj üçün MAK yaradın və yoxlayın.

2. **Fərqli mesajlar:** Eyni açarla iki fərqli mesajın MAC-larını hesablayın.

3. **Uzunluq problemi:** CBC-MAC-ın dəyişən uzunluq problemini nümayiş etdirin.

4. **Həll üsulları:** Bu problemi həll etmək üçün hansı üsullar var?

In [ ]:

# Çalışma 2 - Cavablar

print("📝 ÇALIŞMA 2 CAVABLARI")
print("=" * 80)

# 1-2. CBC-MAC testi
print("\n1-2. CBC-MAC TESTİ:")
print("   Yuxarıdakı nümunədə CBC-MAC yalnız blok-tam (16 baytın qatı) mesaj üçün hesablandı.")

# 3. Uzunluq problemi
print("\n3. UZUNLUQ PROBLEMİ:")
print("""
   CBC-MAC eyni açar altında dəyişən uzunluqlu mesajlar üçün təhlükəsiz deyil.

   M1 və M2 üçün teqlər məlumdursa, aşağıdakı yeni mesaj saxtalaşdırıla bilər:
     F = M1 || (M2_1 ⊕ MAC(M1)) || M2_2 || ...

   Bu halda hücumçu MAC(M2)-ni F üçün etibarlı teq kimi təqdim edə bilər.
""")

# 4. Həll üsulları
print("\n4. HƏLL ÜSULLARI:")
print("""
   CBC-MAC-ın dəyişən uzunluq problemini həll etmək üçün:

   1. CMAC: Əlavə sabitlər (K1, K2) istifadə et
   2. EMAC / ECBC-MAC kimi təkmilləşdirilmiş sxemlərdən istifadə et
   3. HMAC: Heş əsaslı MAK tətbiq et
   4. Eyni açar altında yalnız sabit uzunluqlu mesajları qəbul et

   Praktikada dəyişən uzunluqlu mesajlar üçün CMAC və ya HMAC tövsiyə olunur.
""")


## 🔐 CMAC (Cipher-based MAC) (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔐 CMAC (Cipher-based MAC)</h4>
<p>CMAC, CBC-MAC-ın dəyişən uzunluq problemini həll edən NIST standartıdır (SP 800-38B). Blok-şifr açarından törədilən iki alt açardan istifadə edir:</p>
<ul>
    <li>$K_1$ — tam bloklar üçün</li>
    <li>$K_2$ — qismən bloklar üçün</li>
</ul>
<p>Qeyd: CMAC-də sabit adətən alt açarlar deyil, sonlu sahədə ikiqatlama zamanı istifadə olunan reduksiya dəyəridir; AES-CMAC üçün bu dəyər $R_b = \texttt{0x87}$-dir.</p>
</div>

In [ ]:

class CustomCMAC:
    """
    NIST SP 800-38B əsasında AES-CMAC implementasiyası
    """

    def __init__(self, key):
        if not CRYPTO_AVAILABLE:
            raise ImportError("PyCryptodome tələb olunur")
        if len(key) not in (16, 24, 32):
            raise ValueError("AES açarı 16, 24 və ya 32 bayt olmalıdır")

        self.key = key
        self.block_size = 16
        self.k1, self.k2 = self._generate_subkeys()

    def _dbl(self, block):
        """GF(2^128)-də 2-yə vurma"""
        value = int.from_bytes(block, "big")
        carry = (value >> 127) & 1
        value = ((value << 1) & ((1 << 128) - 1))
        if carry:
            value ^= 0x87
        return value.to_bytes(self.block_size, "big")

    def _generate_subkeys(self):
        # Qeyd: ECB burada yalnız CMAC üçün AES-in tək-blok primitivini çağırmaq
        # məqsədilə istifadə olunur. Mesajların şifrələnməsi üçün ECB təhlükəsiz deyil:
        # eyni açıq mətn blokları eyni şifrmətn blokları yaradır və strukturu sızdırır.
        cipher = AES.new(self.key, AES.MODE_ECB)
        L = cipher.encrypt(b"\x00" * self.block_size)
        k1 = self._dbl(L)
        k2 = self._dbl(k1)
        return k1, k2

    def _pad(self, partial_block):
        """ISO/IEC 7816-4 tipli 10* doldurma"""
        if len(partial_block) >= self.block_size:
            raise ValueError("Yalnız qismən blok doldurula bilər")
        return partial_block + b"\x80" + b"\x00" * (self.block_size - len(partial_block) - 1)

    def generate(self, message):
        """CMAC generasiya et"""
        if not CRYPTO_AVAILABLE:
            return None

        if len(message) == 0:
            blocks = []
            complete_last_block = False
        else:
            blocks = [message[i:i + self.block_size] for i in range(0, len(message), self.block_size)]
            complete_last_block = len(blocks[-1]) == self.block_size

        if complete_last_block:
            last_block = xor_bytes(blocks[-1], self.k1)
            blocks = blocks[:-1] + [last_block]
        else:
            last_partial = blocks[-1] if blocks else b""
            last_block = xor_bytes(self._pad(last_partial), self.k2)
            blocks = blocks[:-1] + [last_block] if blocks else [last_block]

        state = b"\x00" * self.block_size
        # ECB burada şifrələmə rejimi kimi deyil, CMAC zəncirləməsində AES-in
        # deterministik blok çevrilməsi kimi istifadə edilir; real şifrələmə üçün
        # ECB-dən istifadə etməyin.
        cipher = AES.new(self.key, AES.MODE_ECB)

        for block in blocks:
            state = cipher.encrypt(xor_bytes(state, block))

        return state

    def verify(self, message, mac):
        """CMAC yoxla"""
        expected = self.generate(message)
        if expected is None:
            return False
        return hmac.compare_digest(mac, expected)

# Test
if CRYPTO_AVAILABLE:
    print("\n" + "=" * 70)
    print("🔐 CMAC IMPLEMENTASİYASI TESTİ")
    print("=" * 70)

    key = get_random_bytes(16)
    cmac_obj = CustomCMAC(key)

    test_messages = [
        b"",          # boş mesaj - əvvəlki versiyada problemli idi
        b"A" * 16,    # tam blok
        b"B" * 25,    # qismən blok
        b"C" * 32     # iki tam blok
    ]

    for msg in test_messages:
        mac = cmac_obj.generate(msg)
        print(f"📨 Mesaj uzunluğu: {len(msg)} bayt")
        print(f"🔏 MAC: {mac.hex()}")
        print(f"✅ Yoxlama: {cmac_obj.verify(msg, mac)}\n")


### 7.1 PyCryptodome ilə CMAC

In [ ]:

def pycryptodome_cmac_demo():
    """
    PyCryptodome kitabxanası ilə CMAC istifadəsi
    """
    if not CRYPTO_AVAILABLE:
        print("PyCryptodome yüklü deyil")
        return

    print("\n" + "-" * 70)
    print("📦 PYCRYPTODOME İLƏ CMAC")
    print("-" * 70)

    # Açar
    key = get_random_bytes(16)
    print(f"🔑 Açar: {key.hex()}")

    # Mesaj
    message = b"Salam, dunya! Bu CMAC test mesajidir."
    print(f"📨 Mesaj: {message}")

    # Kitabxana CMAC-i
    cmac_lib = PyCryptoCMAC.new(key, ciphermod=AES)
    cmac_lib.update(message)
    lib_mac = cmac_lib.digest()

    # Öz implementasiyamız
    custom_cmac = CustomCMAC(key)
    custom_mac = custom_cmac.generate(message)

    print(f"\n🔏 PyCryptodome CMAC: {lib_mac.hex()}")
    print(f"🔏 Öz CMAC-imiz      : {custom_mac.hex()}")
    print(f"🔁 Uyğundurmu?       : {lib_mac == custom_mac}")

    # Yoxlama
    verifier = PyCryptoCMAC.new(key, ciphermod=AES)
    verifier.update(message)
    try:
        verifier.verify(lib_mac)
        print("\n✅ CMAC uğurla yoxlandı")
    except ValueError:
        print("\n❌ CMAC yoxlanması uğursuz")

    # Dəyişdirilmiş mesaj
    verifier2 = PyCryptoCMAC.new(key, ciphermod=AES)
    verifier2.update(message + b"!")
    try:
        verifier2.verify(lib_mac)
        print("❌ Dəyişdirilmiş mesaj yoxlandı (olmamalıdır!)")
    except ValueError:
        print("✅ Dəyişdirilmiş mesaj uğursuz oldu (düzgün)")

if CRYPTO_AVAILABLE:
    pycryptodome_cmac_demo()


### 7.2 CMAC vs CBC-MAC müqayisəsi

In [ ]:

def cmac_comparison():
    """
    CMAC və CBC-MAC müqayisəsi
    """
    print("\n" + "-" * 70)
    print("📊 CMAC vs CBC-MAC MÜQAYİSƏSİ")
    print("-" * 70)

    print("""
┌──────────────────┬────────────────────────────┬────────────────────────────┐
│ Xüsusiyyət       │ CBC-MAC                     │ CMAC                       │
├──────────────────┼────────────────────────────┼────────────────────────────┤
│ Dəyişən uzunluq  │ ❌ Təhlükəlidir             │ ✅ Davamlı                 │
│ IV               │ 0^128 sabit olmalıdır       │ ✅ Daxili olaraq sabitdir  │
│ Alt açarlar      │ ❌ Yox                      │ ✅ K1, K2                  │
│ Doldurma         │ ❌ Yalnız blok-tam mesajlar │ ✅ Son blok üçün şərti     │
│ Standart         │ Məhdud istifadə             │ ✅ NIST SP 800-38B         │
└──────────────────┴────────────────────────────┴────────────────────────────┘
""")

cmac_comparison()


### ✍️ Çalışma 3: CMAC (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **CMAC testi:** CMAC sinfinizi istifadə edərək müxtəlif uzunluqlu mesajlar üçün MAC yaradın.

2. **Kitabxana müqayisəsi:** PyCryptodome ilə CMAC istifadə edin və nəticələri müqayisə edin.

3. **Sabitlər testi:** Tam blok və qismən blok mesajlar üçün CMAC-ın fərqli sabitlər ($k_1$, $k_2$) istifadə etdiyini yoxlayın.

4. **Üstünlüklər:** CMAC-ın CBC-MAC-dan üstünlüklərini sadalayın.

In [ ]:

# Çalışma 3 - Cavablar

print("📝 ÇALIŞMA 3 CAVABLARI")
print("=" * 80)

if CRYPTO_AVAILABLE:
    import secrets
    from binascii import unhexlify

    # 1. RFC 4493 test vektorları
    print("\n1. RFC 4493 TEST VEKTORLARI:")
    rfc_key = unhexlify("2b7e151628aed2a6abf7158809cf4f3c")
    test_vectors = [
        (b"", "bb1d6929e95937287fa37d129b756746"),
        (unhexlify("6bc1bee22e409f96e93d7e117393172a"), "070a16b46b4d4144f79bdd9dd04a287c"),
        (
            unhexlify(
                "6bc1bee22e409f96e93d7e117393172a"
                "ae2d8a571e03ac9c9eb76fac45af8e51"
                "30c81c46a35ce411"
            ),
            "dfa66747de9ae63030ca32611497c827",
        ),
        (
            unhexlify(
                "6bc1bee22e409f96e93d7e117393172a"
                "ae2d8a571e03ac9c9eb76fac45af8e51"
                "30c81c46a35ce411e5fbc1191a0a52ef"
                "f69f2445df4f9b17ad2b417be66c3710"
            ),
            "51f0bebf7e3b9d92fc49741779363cfe",
        ),
    ]

    all_ok = True
    custom_cmac = CustomCMAC(rfc_key)

    for i, (msg, expected_hex) in enumerate(test_vectors, start=1):
        got = custom_cmac.generate(msg).hex()
        ok = got == expected_hex
        all_ok &= ok
        print(f"   Vektor {i}: {ok} | alınan={got}")

    print(f"   Ümumi nəticə: {all_ok}")

    # 2. Kitabxana ilə müqayisə
    print("\n2. PyCryptodome MÜQAYİSƏSİ:")
    comparison_ok = True
    for _ in range(20):
        key = get_random_bytes(16)
        msg = get_random_bytes(secrets.randbelow(65))

        custom_cmac = CustomCMAC(key)
        custom_mac = custom_cmac.generate(msg)

        lib_cmac = PyCryptoCMAC.new(key, ciphermod=AES)
        lib_cmac.update(msg)
        lib_mac = lib_cmac.digest()

        if custom_mac != lib_mac:
            comparison_ok = False
            break

    print(f"   20 təsadüfi testin hamısı uyğun gəldi? {comparison_ok}")

# 3. Sabitlər testi
print("\n3. SABİTLƏR TESTİ:")
print("""
   Tam blok üçün son blok K1 ilə XOR edilir:
     M_n = M_n ⊕ K1

   Qismən və ya boş mesaj üçün son blok pad(M_n) ⊕ K2 olur:
     M_n = pad(M_n) ⊕ K2

   Bu dəyişiklik CBC-MAC-dəki uzunluq hücumlarını aradan qaldırır.
""")

# 4. Üstünlüklər
print("\n4. CMAC ÜSTÜNLÜKLƏRİ:")
print("""
   CMAC-ın CBC-MAC-dan üstünlükləri:
   • Dəyişən uzunluqlu mesajlar üçün təhlükəsizdir
   • Boş mesajı da düzgün emal edir
   • K1 və K2 sabitləri ilə son bloku ayrıca emal edir
   • NIST SP 800-38B standartına uyğundur
""")


## 🔢 GMAC (GCM Authentication) (10 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔢 GMAC (Galois Message Authentication Code)</h4>
<p>GMAC, GCM (Galois/Counter Mode) rejiminin autentifikasiya hissəsidir. Yalnız autentifikasiya təmin edir (şifrləmə yox).</p>

<p><b>Xüsusiyyətlər:</b></p>
<ul>
    <li>$\text{GF}(2^{128})$ Qalua meydanında işləyir</li>
    <li>GHASH funksiyası əsasında qurulub</li>
    <li>Çox sürətli (paralelləşdirilə bilər)</li>
    <li>AES-GCM TLS 1.2/1.3-də istifadə olunur; GMAC isə GCM-in autentifikasiya komponenti kimi əlaqəlidir</li>
</ul>
</div>

In [ ]:

MIN_GMAC_TAG_LEN = 12  # 96 bit


def gmac_tag(key, nonce, authenticated_data, mac_len=16):
    """
    GMAC teqi hesabla.

    authenticated_data: şifrələnməyən, amma autentifikasiya olunan data (AAD)
    """
    if mac_len < MIN_GMAC_TAG_LEN:
        raise ValueError("GMAC tag ən azı 96 bit (12 bayt) olmalıdır")

    if not CRYPTO_AVAILABLE:
        return None

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce, mac_len=mac_len)
    if authenticated_data:
        cipher.update(authenticated_data)
    return cipher.digest()


def gmac_verify(key, nonce, authenticated_data, tag):
    """GMAC teqini yoxla."""
    if len(tag) < MIN_GMAC_TAG_LEN:
        raise ValueError("GMAC tag ən azı 96 bit (12 bayt) olmalıdır")

    if not CRYPTO_AVAILABLE:
        return False

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce, mac_len=len(tag))
    if authenticated_data:
        cipher.update(authenticated_data)
    try:
        cipher.verify(tag)
        return True
    except ValueError:
        return False


def gmac_demo():
    """
    GMAC nümayişi (GCM-in autentifikasiya hissəsi)
    """
    if not CRYPTO_AVAILABLE:
        return

    print("\n" + "-" * 70)
    print("🔢 GMAC (GALOIS MESSAGE AUTHENTICATION CODE)")
    print("-" * 70)

    key = get_random_bytes(16)
    nonce = get_random_bytes(12)
    authenticated_data = b"header: authenticated only"

    print(f"🔑 Açar: {key.hex()}")
    print(f"🔢 Nonce: {nonce.hex()}")
    print(f"📨 Autentifikasiya olunan data: {authenticated_data}")

    tag = gmac_tag(key, nonce, authenticated_data)

    print(f"\n🔏 GMAC: {tag.hex()}")
    print(f"✅ Yoxlama: {gmac_verify(key, nonce, authenticated_data, tag)}")
    print(f"❌ Dəyişdirilmiş data yoxlaması: {gmac_verify(key, nonce, authenticated_data + b'!', tag)}")

    # Nonce fərqini göstər
    nonce2 = get_random_bytes(12)
    tag2 = gmac_tag(key, nonce2, authenticated_data)
    print(f"\n🔁 Fərqli nonce -> fərqli GMAC? {tag != tag2}")

if CRYPTO_AVAILABLE:
    gmac_demo()


### ✍️ Çalışma 4: GMAC (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **GMAC yaratma:** GMAC ilə bir mesaj üçün autentifikasiya teqi yaradın.

2. **Nonce testi:** Eyni açar, fərqli nonce ilə GMAC hesablayın. Nəticələr fərqlidirmi?

3. **CMAC vs GMAC:** GMAC-ın CMAC-dan fərqləri nələrdir?

4. **Protokol tətbiqləri:** GMAC hansı protokollarda istifadə olunur?

In [ ]:
# Çalışma 4 - Cavablar

print("📝 ÇALIŞMA 4 CAVABLARI")
print("=" * 80)

# 1. GMAC yaratma
print("\n1. GMAC YARATMA:")
print("   Yuxarıdakı nümunədə AAD (şifrələnməyən data) üçün GMAC teqi yaradıldı və yoxlandı.")

# 2. Nonce testi
print("\n2. NONCE TESTİ:")
print("""
   GMAC-də nonce təkrarlanmamalıdır:
   • Fərqli nonce → fərqli GMAC
   • Eyni açarla nonce/IV heç vaxt təkrarlanmamalıdır; təkrar GMAC/GCM-də
     autentifikasiya açarı haqqında məlumat sızdıra və saxta teqlərin
     yaradılmasına imkan verə bilər.
""")

# 3. CMAC vs GMAC
print("\n3. CMAC vs GMAC:")
print("""
   CMAC:
   • Blok şifrə əsaslı MAK-dir
   • CBC-MAC-in təkmilləşdirilmiş versiyasıdır
   • K1 və K2 sabitlərindən istifadə edir

   GMAC:
   • GCM-in yalnız autentifikasiya hissəsidir
   • AAD-ni autentifikasiya edir, amma şifrələmir
   • GF(2^128) Qalua meydanında GHASH istifadə edir
""")

# 4. Protokol tətbiqləri
print("\n4. PROTOKOL TƏTBİQLƏRİ:")
print("""
   GMAC/GCM ailəsi istifadə olunan protokollar:
   • TLS 1.2/1.3 (AES-GCM)
   • IPsec
   • IEEE 802.1AE (MACsec)
   • WPA3 (GCMP)
""")


## ⚖️ MAK və rəqəmsal imza müqayisəsi (10 dəq)

In [ ]:
def mac_vs_signature_comparison():
    """
    MAK və rəqəmsal imza müqayisəsi
    """
    print("\n" + "-" * 70)
    print("⚖️ MAK vs RƏQƏMSAL İMZA MÜQAYİSƏSİ")
    print("-" * 70)

    print("""
┌──────────────────┬────────────────────────────┬────────────────────────────┐
│ Xüsusiyyət       │ MAK                         │ Rəqəmsal imza              │
├──────────────────┼────────────────────────────┼────────────────────────────┤
│ Açar növü        │ Simmetrik (eyni açar)       │ Asimmetrik (açıq/gizli açar)│
│ Tərəf sayı       │ 2 tərəf                     │ Çox tərəf                  │
│ İnkar edilməzlik │ ❌ Xeyr                     │ ✅ Bəli                    │
│ Sürət            │ ✅ Çox sürətli              │ ❌ Yavaş                   │
│ Açar paylanması  │ ❌ Çətin (təhlükəsiz kanal)  │ ✅ Asan (açıq açar)        │
│ Nümunələr        │ HMAC, CMAC, GMAC           │ RSA, ECDSA, Ed25519        │
└──────────────────┴────────────────────────────┴────────────────────────────┘
""")

mac_vs_signature_comparison()

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50;">
<h4>📌 Nə vaxt MAK, nə vaxt rəqəmsal imza?</h4>
<ul>
    <li><b>MAK:</b> İki tərəf arasında təhlükəsiz kanal varsa, sürət tələb olunursa</li>
    <li><b>Rəqəmsal imza:</b> Üçüncü tərəflərə sübut etmək lazımdırsa, inkar edilməzlik tələb olunursa</li>
</ul>
</div>

## 🛡️ UF-CMA təhlükəsizlik modeli (10 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🛡️ UF-CMA (Unforgeability under Chosen Message Attacks)</h4>
<p>MAK-ın təhlükəsizlik modeli. Hücumçu:</p>
<ol>
    <li>İstədiyi mesajlar üçün MAC sorğu edə bilər</li>
    <li>Yeni mesaj üçün etibarlı MAC yaratmalıdır</li>
</ol>
</div>

In [ ]:
def uf_cma_simulation():
    """
    UF-CMA təhlükəsizlik modelinin simulyasiyası
    """
    if not CRYPTO_AVAILABLE:
        return

    print("\n" + "-" * 70)
    print("🛡️ UF-CMA TƏHLÜKƏSİZLİK MODELİ")
    print("-" * 70)

    key = get_random_bytes(16)

    print("Hücumçu seçdiyi mesajlar üçün MAC sorğu edə bilər:")
    queries = [
        b"message1",
        b"message2",
        b"message3"
    ]

    macs = {}
    for msg in queries:
        cmac_obj = PyCryptoCMAC.new(key, ciphermod=AES)
        cmac_obj.update(msg)
        macs[msg] = cmac_obj.digest()
        print(f"  MAC({msg}) = {macs[msg].hex()}")

    print("\nHücumçu yeni mesaj üçün etibarlı MAC yaratmalıdır:")
    new_message = b"message4"

    # Həqiqi MAC
    cmac_obj = PyCryptoCMAC.new(key, ciphermod=AES)
    cmac_obj.update(new_message)
    real_mac = cmac_obj.digest()
    print(f"✅ Həqiqi MAC({new_message}) = {real_mac.hex()}")

    # Hücumçu təsadüfi MAC təxmin edir
    forged_mac = get_random_bytes(16)
    print(f"🕵️ Saxta MAC təxmini = {forged_mac.hex()}")

    probability = 1 / (2**128)
    print(f"\n📊 Bir təsadüfi təxmin üçün uğur ehtimalı (128-bit tag): {probability:.2e}")
    print("   Qeyd: UF-CMA təhlükəsizliyi yalnız təsadüfi təxmin etməni deyil,")
    print("   istənilən effektiv hücumçunun saxtalaşdırma üstünlüyünü məhdudlaşdırır.")
    print("   Bu üstünlük sorğuların/verifikasiya cəhdlərinin sayı, tag uzunluğu və MAC konstruksiyasından asılıdır.")

if CRYPTO_AVAILABLE:
    uf_cma_simulation()


### ✍️ Çalışma 5: Təhlükəsizlik modelləri (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **UF-CMA izahı:** UF-CMA təhlükəsizlik modelini izah edin.

2. **MAK vs İmza:** MAK ilə rəqəmsal imza arasında 3 əsas fərqi sadalayın.

3. **İnkar edilməzlik:** Niyə MAK inkar edilməzlik təmin etmir?

4. **Seçim kriteriyaları:** Hansı hallarda MAK, hansı hallarda rəqəmsal imza istifadə edilməlidir?

In [ ]:
# Çalışma 5 - Cavablar

print("📝 ÇALIŞMA 5 CAVABLARI")
print("=" * 80)

# 1. UF-CMA izahı
print("\n1. UF-CMA İZAHI:")
print("""
   UF-CMA (Unforgeability under Chosen Message Attacks):
   • Hücumçu istədiyi mesajlar üçün MAC sorğu edə bilər
   • Bu sorğular əsasında yeni, sorğulanmamış bir mesaj üçün
     etibarlı MAC yaratmağa çalışır
   • Təhlükəsiz MAK üçün hücumçunun uğur ehtimalı nəzərə alınmayacaq qədər kiçik olmalıdır
""")

# 2. MAK vs İmza (artıq yuxarıda cədvəl var)
print("\n2. MAK VS İMZA FƏRQLƏRİ:")
print("""
   1) Açar növü: MAK simmetrik (eyni açar), imza asimmetrik
   2) İnkar edilməzlik: MAK təmin etmir, imza təmin edir
   3) Tərəf sayı: MAK 2 tərəf arasında, imza çox tərəfə sübut oluna bilər
""")

# 3. İnkar edilməzlik
print("\n3. İNKAR EDİLMƏZLİK:")
print("""
   MAK inkar edilməzlik təmin etmir, çünki:
   • MAK simmetrik açarla yaradılır - eyni açar hər iki tərəfdə var
   • Həm Alis, həm də Bob eyni MAK-ı yarada bilər
   • Üçüncü tərəf MAK-ı kimin yaratdığını müəyyən edə bilməz

   Rəqəmsal imza asimmetrik açarla yaradılır:
   • Yalnız gizli açar sahibi imza yarada bilər
   • Açıq açar hər kəsə yoxlamağa imkan verir
""")

# 4. Seçim kriteriyaları
print("\n4. SEÇİM KRİTERİYALARI:")
print("""
   MAK seçilməlidir:
   • İki tərəf arasında təhlükəsiz kanal varsa
   • Yüksək sürət tələb olunursa
   • İnkar edilməzlik tələb olunmursa

   Rəqəmsal imza seçilməlidir:
   • Üçüncü tərəflərə sübut etmək lazımdırsa
   • İnkar edilməzlik tələb olunursa
   • Açarların təhlükəsiz paylanması çətindirsə
""")

## 🖥️ İnteqrasiya edilmiş tətbiq (15 dəq)

In [ ]:

def mac_lab_menu():
    """
    MAK laboratoriyası interaktiv menyu
    """
    while True:
        print("\n" + "=" * 70)
        print("🔏 MAK LABORATORİYASI - Məşğələ 16")
        print("=" * 70)
        print("1. 🔏 HMAC-SHA256 hesabla")
        print("2. 🔗 CBC-MAC nümayişi")
        print("3. 🔐 CMAC (öz implementasiya)")
        print("4. 📦 PyCryptodome CMAC")
        print("5. 🔢 GMAC nümayişi")
        print("6. ⚖️ MAK vs Rəqəmsal imza müqayisəsi")
        print("7. 🛡️ UF-CMA təhlükəsizlik modeli")
        print("0. ❌ Çıxış")
        print("=" * 70)

        choice = input("📌 Seçiminiz: ")

        if choice == '1':
            message = input("📨 Mesaj: ").encode()
            key = input("🔑 Açar: ").encode()
            if len(key) == 0:
                key = get_random_bytes(16)
                print(f"Təsadüfi açar: {key.hex()}")

            mac = hmac_sha256(key, message)
            print(f"\n🔏 HMAC-SHA256: {mac.hex()}")
            print(f"✅ Yoxlama: {hmac_verify(key, message, mac)}")

        elif choice == '2' and CRYPTO_AVAILABLE:
            message = input("📨 Mesaj: ").encode()
            if len(message) == 0:
                print("❌ CBC-MAC üçün boş mesaj istifadə etməyin")
                continue

            # Nəzəri CBC-MAC sabit uzunluq tələb edir; demo üçün blok ölçüsünə qədər doldururuq
            padded_message = pad(message, 16)
            key = get_random_bytes(16)

            mac = cbc_mac(padded_message, key)
            print(f"🔑 Açar: {key.hex()}")
            print(f"📦 Doldurulmuş mesaj uzunluğu: {len(padded_message)} bayt")
            print(f"🔏 CBC-MAC: {mac.hex()}")

            print("\n⚠️ Qeyd: CBC-MAC eyni açar altında yalnız sabit uzunluqlu mesajlar üçün təhlükəsizdir.")
            cbc_mac_length_extension_attack()

        elif choice == '3' and CRYPTO_AVAILABLE:
            message = input("📨 Mesaj: ").encode()
            key = get_random_bytes(16)

            cmac_obj = CustomCMAC(key)
            mac = cmac_obj.generate(message)

            print(f"🔑 Açar: {key.hex()}")
            print(f"🔏 CMAC: {mac.hex()}")
            print(f"✅ Yoxlama: {cmac_obj.verify(message, mac)}")

        elif choice == '4' and CRYPTO_AVAILABLE:
            pycryptodome_cmac_demo()

        elif choice == '5' and CRYPTO_AVAILABLE:
            gmac_demo()

        elif choice == '6':
            mac_vs_signature_comparison()

        elif choice == '7' and CRYPTO_AVAILABLE:
            uf_cma_simulation()

        elif choice == '0':
            print("👋 Proqram bitdi. Sağ olun!")
            break

        else:
            print("❌ Yanlış seçim və ya kitabxana yüklü deyil")

# Proqramı işə sal (istəyə bağlı)
# mac_lab_menu()


## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: MAK paketi (3 bal)

Aşağıdakı funksiyaları özündə birləşdirən Python paketi yazın:

```
mac_package/
├── __init__.py
├── hmac_demo.py        # HMAC-SHA256 hesablamaları
├── cbc_mac.py          # CBC-MAC implementasiyası, uzunluq problemi nümayişi
├── cmac.py             # CMAC implementasiyası (öz sinfiniz)
├── gmac.py             # GMAC nümayişi
├── security_models.py  # UF-CMA simulyasiyası
└── main.py             # Bütün funksiyaları birləşdirən interaktiv menyu
```

**Tələblər:**
* Hər bir funksiya üçün docstring yazın
* Səhv hallarını idarə edin (try-except)
* Kod təmiz və oxunaqlı olmalıdır
* Hər modul üçün test nümunələri əlavə edin

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **Fayl MAK-ı:** HMAC-SHA256 istifadə edərək bir faylın MAK-nı hesablayın.

2. **CBC-MAC hücumu:** CBC-MAC-ın dəyişən uzunluq problemini nümayiş etdirən proqram yazın.

3. **CMAC testi:** CMAC implementasiyanızı PyCryptodome-un CMAC-ı ilə müqayisə edin.

4. **GMAC teqi:** GMAC ilə bir mesajın autentifikasiya teqini yaradın.

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və aşağıdakı suallara cavab tapın. Cavablarınızı 1-2 səhifəlik hesabat şəklində təqdim edin:

1. **NIST SP 800-38B:** NIST SP 800-38B standartı haqqında məlumat toplayın. CMAC-ın spesifikasiyası nədir?

2. **CBC-MAC problemi:** CBC-MAC-ın dəyişən uzunluq probleminin riyazi izahını verin. Niyə bu problem yaranır?

3. **GMAC GHASH:** GMAC-ın GHASH funksiyası necə işləyir? GF(2¹²⁸) arifmetikasını araşdırın.

4. **MAK vs İmza:** MAK və rəqəmsal imza arasında seçim edərkən hansı amillər nəzərə alınmalıdır? Real dünya nümunələri verin.

**Format tələbləri:**
* PDF formatında təqdim edin
* Ən azı 5 qaynaq göstərin (kitab, məqalə, veb səhifə)
* Öz sözlərinizlə yazın (copy-paste yox)
* Mümkünsə, sxemlər və ya diaqramlar əlavə edin

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<p>Bu məşğələdə öyrəndiklərimiz:</p>
<ul>
    <li>✅ <b>MAK:</b> Mesajın tamlığını və mənbə autentifikasiyasını təmin edir</li>
    <li>✅ <b>CBC-MAC:</b> Sabit uzunluqlu mesajlar üçün təhlükəsiz, dəyişən uzunluqda problem var</li>
    <li>✅ <b>CMAC:</b> CBC-MAC-ın problemlərini həll edən NIST standartı ($k_1$, $k_2$ sabitləri)</li>
    <li>✅ <b>GMAC:</b> GCM rejiminin autentifikasiya hissəsi, Qalua meydanında işləyir</li>
    <li>✅ <b>UF-CMA:</b> MAK-ın təhlükəsizlik modeli</li>
    <li>✅ <b>MAK vs imza:</b> MAK simmetrik, sürətli; imza asimmetrik, inkar edilməzlik təmin edir</li>
</ul>
</div>

### 💭 Müzakirə sualları

1. Bu məşğələdə ən maraqlı tapdığınız nə oldu?
2. CBC-MAC-ın dəyişən uzunluq problemini necə həll etmək olar?
3. CMAC-ın CBC-MAC-dan üstünlükləri nələrdir?
4. GMAC nə üçün CMAC-dan sürətlidir?
5. Hansı hallarda MAK, hansı hallarda rəqəmsal imza istifadə etməlisiniz?
6. UF-CMA modelində hücumçu nə edə bilər?
7. Niyə CMAC uzunluq genişləndirmə hücumlarına qarşı davamlıdır?

## 📚 Əlavə resurslar

* 📘 **NIST SP 800-38B (CMAC):** [https://csrc.nist.gov/pubs/sp/800/38/b/upd1/final](https://csrc.nist.gov/pubs/sp/800/38/b/upd1/final) — NIST bu sənədin yenilənməsi/reviziyası üzərində işləməyi planlaşdırır.
* 📙 **NIST SP 800-38D (GCM/GMAC):** [https://csrc.nist.gov/pubs/sp/800/38/d/final](https://csrc.nist.gov/pubs/sp/800/38/d/final) — NIST SP 800-38D sənədini reviziya etməyi qərara alıb.
* 📗 **RFC 4493 (AES-CMAC):** [https://www.rfc-editor.org/rfc/rfc4493.html](https://www.rfc-editor.org/rfc/rfc4493.html)
* 📕 **PyCryptodome CMAC:** [https://pycryptodome.readthedocs.io/en/latest/src/hash/cmac.html](https://pycryptodome.readthedocs.io/en/latest/src/hash/cmac.html)
* 📘 **Python hmac modulu:** [https://docs.python.org/3/library/hmac.html](https://docs.python.org/3/library/hmac.html)
* 📙 **Cryptography CMAC:** [https://cryptography.io/en/latest/hazmat/primitives/mac/cmac/](https://cryptography.io/en/latest/hazmat/primitives/mac/cmac/)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 16 tamamlandı!</h2>
<p>Bütün kodları və tapşırıq cavablarını növbəti məşğələyə qədər təqdim edin.</p>
<p><em>Kodlar aydın şərhlərlə yazılmalı və asan oxunmalıdır. Hər bir funksiyanın nə etdiyini izah edən şərhlər əlavə edin.</em></p>
<p style="font-size: 1.2em; margin-top: 15px;">🔏 <b>MAK - mesaj tamlığının qarantı!</b></p>
</div>
